# SC4001: Few-Shot Flower Recognition Evaluation

This notebook evaluates both `ViT-VPT-Shallow` and `ViT-VPT-Deep` checkpoints on the test set and compares their performance.

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath("__file__"))))

import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from tqdm.notebook import tqdm

from models.vit_model import ViTForFlowerRecognition
from models.vit_model_deep import ViTDeepForFlowerRecognition
from utils.data_loader import get_dataloaders

### 1. Load Data and Models

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

_, _, test_loader = get_dataloaders(batch_size=32)

def load_model(model, checkpoint_path, device):
    try:
        model.load_state_dict(torch.load(checkpoint_path, map_location=device))
        print(f"Loaded: {checkpoint_path}")
    except FileNotFoundError:
        print(f"Not found: {checkpoint_path}")
    return model.to(device).eval()

shallow_model = load_model(ViTForFlowerRecognition(num_classes=102, num_prompts=10), 'results/best_vit_vpt_model.pth', device)
deep_model    = load_model(ViTDeepForFlowerRecognition(num_classes=102, num_prompts=10), 'results/best_vit_vpt_deep_model.pth', device)

### 2. Run Inference on Test Set

In [3]:
def run_inference(model, dataloader, device, desc="Evaluating"):
    all_preds, all_targets = [], []
    with torch.no_grad():
        for inputs, targets in tqdm(dataloader, desc=desc):
            inputs, targets = inputs.to(device), targets.to(device)
            logits, _ = model(inputs)
            _, predicted = torch.max(logits.data, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())
    return np.array(all_preds), np.array(all_targets)

shallow_preds, targets = run_inference(shallow_model, test_loader, device, "VPT-Shallow")
deep_preds, _          = run_inference(deep_model,    test_loader, device, "VPT-Deep")

shallow_acc = np.mean(shallow_preds == targets)
deep_acc    = np.mean(deep_preds    == targets)

print(f"VPT-Shallow Test Accuracy: {shallow_acc * 100:.2f}%")
print(f"VPT-Deep    Test Accuracy: {deep_acc    * 100:.2f}%")

VPT-Shallow:   0%|          | 0/193 [00:00<?, ?it/s]

VPT-Deep:   0%|          | 0/193 [00:00<?, ?it/s]

VPT-Shallow Test Accuracy: 85.28%
VPT-Deep    Test Accuracy: 91.72%


### 3. Accuracy Comparison

In [ ]:
models_names = ['VPT-Shallow', 'VPT-Deep']
accuracies   = [shallow_acc * 100, deep_acc * 100]

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(models_names, accuracies, color=['steelblue', 'darkorange'], width=0.4)
ax.set_ylim(0, 100)
ax.set_ylabel('Test Accuracy (%)')
ax.set_title('VPT-Shallow vs VPT-Deep — Oxford Flowers 102')
for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5, f'{acc:.2f}%', ha='center', fontsize=12)
plt.tight_layout()
plt.savefig('results/accuracy_comparison.png', dpi=150)
plt.show()

### 4. Classification Reports

In [ ]:
for name, preds in [('VPT-Shallow', shallow_preds), ('VPT-Deep', deep_preds)]:
    report = classification_report(targets, preds, zero_division=0)
    print(f"=== {name} ===")
    print(report)
    fname = f"results/classification_report_{name.lower().replace('-', '_')}.txt"
    with open(fname, "w") as f:
        f.write(report)
    print(f"Saved {fname}\n")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(40, 16))

for ax, name, preds in zip(axes, ['VPT-Shallow', 'VPT-Deep'], [shallow_preds, deep_preds]):
    cm = confusion_matrix(targets, preds)
    sns.heatmap(cm, annot=False, cmap="Blues", cbar=True, ax=ax)
    ax.set_title(f"{name} — Confusion Matrix (102 Classes)", fontsize=18)
    ax.set_xlabel("Predicted Label", fontsize=14)
    ax.set_ylabel("True Label", fontsize=14)

plt.tight_layout()
plt.savefig("results/confusion_matrix_comparison.png", dpi=200)
plt.show()

### 5. Confusion Matrices